# Multi-source, question-focused summarizer — **LangGraph**

This is one of four notebooks that build the **same** small agentic app, each in a different
framework, so you can compare how they wire up identical ideas. Here it's **LangGraph**.

**The task.** You give the agent either a **list of URLs** *or* a **local file path**, plus a
**question**. The agent **decides on its own** how to fetch, parse, and summarize — then returns a
summary focused on your question.

```
            ┌─────────────────────────── the agent decides ───────────────────────────┐
 question + │  URL?  ──▶  fetch_url  (scraper, an MCP server)  ──▶  parse_html (tool)   │
 sources ──▶│  .pdf? ──▶  parse_pdf  (tool)                                             │──▶ summarize (tool) ──▶ answer
            └──────────────────────────────────────────────────────────────────────────┘
```

**What is shared vs. what differs.** The parsers and the summarizer are plain Python tools imported
from `src/` — *identical* in every notebook. The **scraper is exposed as an MCP server** (a separate
process), so each framework has to show how it *consumes* MCP. The only thing that changes between
notebooks is the framework-specific wiring below: how it builds the model, registers tools, connects
the MCP server, runs the agent, and handles memory. Read those side by side.

## 1. Install

Everything is declared in `pyproject.toml`; install the project (editable) plus dev tools. If you
already ran `uv pip install -e ".[dev]"` in your venv, this is a no-op.

To use a **local** model instead of OpenAI, start Ollama first: `ollama serve` and
`ollama pull qwen3.5:4b` (see the project README).

In [ ]:
%pip install -q -e ".[dev]"

## 2. Choose the model provider

One env var flips the whole notebook between **OpenAI** (`gpt-5-mini`) and a **local Ollama** model
(`qwen3.5:4b`, served on an OpenAI-compatible endpoint). `load_llm_config()` reads it and returns a
small config object that both the summarizer tool and the agent below are built from.

In [ ]:
import os

from dotenv import load_dotenv

from src.config import load_llm_config

load_dotenv()  # reads .env (OPENAI_API_KEY, etc.)

# Flip this to "ollama" to run fully locally (requires `ollama serve` + the pulled model).
os.environ["LLM_PROVIDER"] = "openai"

cfg = load_llm_config()  # reads LLM_PROVIDER at call time, so the line above takes effect
print(f"provider={cfg.provider!r}  model={cfg.model!r}  base_url={cfg.base_url!r}")

## 3. The shared tools (and why the scraper is an MCP server)

Three of the four tools are ordinary in-process Python functions, imported from `src/` and reused
**unchanged** by every framework:

- `html_to_text(html)` — strip boilerplate from raw HTML and return clean text.
- `pdf_to_text(path)` — extract clean text from a local PDF.
- `summarize_for_question(text, question)` — a question-focused summary (LLM call, works for both providers).

The **scraper** is different: it runs as a **separate process** and is reached over the **Model
Context Protocol (MCP)** via stdio. This is the realistic shape for an external capability — a tool
the agent talks to over a protocol rather than calling in-process. Each framework section shows how
it launches and consumes that server (the spawn command is the same everywhere).

In [ ]:
import os
import sys

from src.tools.parsers.html_parser import html_to_text
from src.tools.parsers.pdf_parser import pdf_to_text
from src.tools.summarizer import summarize_for_question

# How every framework spawns the scraper MCP server (stdio subprocess):
REPO_ROOT = os.getcwd()  # notebooks run from the repo root
SCRAPER_CMD = sys.executable  # this venv's python
SCRAPER_ARGS = ["-m", "src.mcp_servers.scraper_server"]
print("scraper launch:", SCRAPER_CMD, *SCRAPER_ARGS)

# Example inputs reused by the run cells below (swap in your own URL / question / PDF):
EXAMPLE_URL = "https://en.wikipedia.org/wiki/Large_language_model"
EXAMPLE_Q = "What are the main capabilities and risks of large language models?"

## 4. Sanity check the MCP server

Before wiring it into an agent, confirm the server starts and advertises its tools. This spawns the
server over stdio and lists what the agent will receive (`fetch_url`, `fetch_urls`). You can also run
it standalone in a terminal: `python -m src.mcp_servers.scraper_server`.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def list_scraper_tools() -> list[str]:
    params = StdioServerParameters(
        command=SCRAPER_CMD, args=SCRAPER_ARGS, cwd=REPO_ROOT, env=dict(os.environ)
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            return [t.name for t in tools.tools]


await list_scraper_tools()

## 5. LangGraph

LangGraph thinks in graphs. `create_react_agent` wires the reason→act→observe loop for you. Tools are
`@tool`-decorated functions; the MCP scraper's tools are loaded through **`langchain-mcp-adapters`**
(`MultiServerMCPClient.get_tools()` — async) and merged with the in-process tools. A `MemorySaver`
checkpointer plus a `thread_id` give **real persistence** and multi-turn essentially for free.

The model is a `ChatOpenAI`; pointing `base_url` at Ollama switches it to local.

**Build the model and the in-process tools.** (MCP tools are loaded in the next cell because that call is async.)

In [ ]:
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent


def build_model(cfg):
    # base_url=None -> hosted OpenAI; ChatOpenAI handles gpt-5* params automatically.
    return ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key)


@tool
def parse_html(html: str) -> str:
    """Convert raw HTML (from fetch_url) into clean text."""
    return html_to_text(html)


@tool
def parse_pdf(path: str) -> str:
    """Extract clean text from a local PDF path."""
    return pdf_to_text(path)


@tool
def summarize(text: str, question: str) -> str:
    """Return a question-focused summary of text."""
    return summarize_for_question(text, question)

In [ ]:
# Load the scraper's MCP tools (async) and assemble the ReAct agent.
mcp_client = MultiServerMCPClient(
    {
        "scraper": {
            "command": SCRAPER_CMD,
            "args": SCRAPER_ARGS,
            "transport": "stdio",
            "cwd": REPO_ROOT,
            "env": dict(os.environ),
        }
    }
)
mcp_tools = await mcp_client.get_tools()

PROMPT = (
    "You answer a question by summarizing sources. For a local .pdf path use parse_pdf; "
    "for a URL use fetch_url then parse_html; then use summarize. Stay grounded in the content."
)
agent = create_react_agent(
    build_model(cfg),
    tools=[*mcp_tools, parse_html, parse_pdf, summarize],
    prompt=PROMPT,
    checkpointer=MemorySaver(),  # persistence keyed by thread_id
)


async def answer(sources: str, question: str, thread_id: str = "summarizer-1") -> str:
    run_cfg = {"configurable": {"thread_id": thread_id}}
    out = await agent.ainvoke(
        {"messages": [("user", f"Sources: {sources}\nQuestion: {question}")]}, run_cfg
    )
    return out["messages"][-1].content


# Example A — a URL: expect fetch_url -> parse_html -> summarize
print(await answer(EXAMPLE_URL, EXAMPLE_Q))

In [ ]:
# Example B — a local file: expect parse_pdf -> summarize
if os.path.exists("data/sample.pdf"):
    print(await answer("data/sample.pdf", "Summarize the conclusion.", thread_id="pdf"))
else:
    print("Put a PDF at data/sample.pdf to try the local-file path.")

**Memory.** LangGraph genuinely persists: the `MemorySaver` checkpointer keeps state per
`thread_id`, so re-using the same id continues the conversation. `MemorySaver` is in-process; for
persistence across restarts swap in a disk checkpointer (e.g. `langgraph-checkpoint-sqlite`) or add a
`store=...`. To seed a persistent *hint*, write it to a `store` before the run.

## 6. Step back: the same app, four wirings

You've now seen this framework end to end. Compared with the others:

- **Pydantic AI** — typed and compact; tools are decorated functions, the MCP server is a `toolset`,
  multi-turn is explicit (`message_history`). No built-in persistent store.
- **LangGraph** — a graph with a prebuilt ReAct loop; MCP tools arrive via an adapter; real
  persistence through a `checkpointer` + `thread_id`.
- **OpenAI Agents SDK** — the leanest surface; native `mcp_servers=[...]`; the cleanest *persistent*
  memory via `SQLiteSession` (writes to disk).
- **Google ADK** — code-first around `LlmAgent` + `Runner`; MCP via `MCPToolset`; memory through
  session state, with a `{routing_hint}` injected into the instruction.

Same parsers, same summarizer, same MCP scraper — only the orchestration differs. That difference is
the thing this lesson is about.